# Phase 0: basic metrics for every trained model (VisDrone val + test-dev)

Supervisor's request: proof that each model was trained, plus its basic metrics.
Everything goes through **dronekit** (embedded below), the one implementation of the evaluation
protocol: 512 px slices resized to 1280 + whole image at 1536, boundary filter,
class-wise NMS 0.6, conf ≥ 0.01, COCO AP at maxDets 500 with ignore regions as crowd boxes.

**Models:** YOLO26m-Tile, YOLOE-26m-FT, YOLOE-26m-LP, YOLOE-26m-ZS (descriptive prompts, the val-chosen set from
`yoloe-1`), YOLO26s (only if its weights are attached), plus the Ensemble (YOLO26m-Tile + YOLOE-26m-FT, WBF 0.65).

## How to run

**Inputs:** Add Input → Your Work → Notebooks, then attach the outputs of
1. `visdrone-training` (YOLO26m-Tile: `runs/K1_y26m_tiles/…`)
2. `yoloe-2` (YOLOE-26m-FT: `yoloe26m_e2_full.pt`, `runs_yoloe/e2_full/…`)
3. `yoloe-1` (YOLOE-26m-LP: `yoloe26m_e1_linprobe.pt`, `runs_yoloe/e1_linprobe/…`)
4. `evaluation` (`eval_v2/results_v2.csv` and `eval_v2/cache/*.pkl`, used only for the regression check)
5. *(optional)* the YOLO26s weights. Set `YOLO26S_PATTERN` in the Config cell to their filename.

Files are found by name with `rglob`, so folder layout doesn't matter. Settings: **Internet ON**.

**Step 1: CPU check (about 15 min, no GPU hours used).** Accelerator **None** → *Save Version → Save & Run All*.
The unit tests, the regression check against the old eval-v2 caches and the 5-image smoke test run;
the full run is skipped automatically because there is no GPU. Send back the log if anything fails.

**Step 2: full run (about 2–2.5 h).** Accelerator **GPU T4 x2** → *Save Version → Save & Run All*.
Only GPU 0 is used, so latencies are single-T4 numbers.

**Output:** everything is in `/kaggle/working/phase0/` (results CSVs, figures, detection caches, weights, run files,
code). From the finished version's Output tab, make one **private Dataset** from it (*New Dataset*) and share
it with the second account. Phases 3–4 read the caches from there, so no detector is re-run on VisDrone.
Download `phase0/results/REPORT_tables.md` and the log of the last cells.

In [ ]:
# --- Install (Kaggle) ------------------------------------------------------
from pathlib import Path
if Path("/kaggle").exists():
    !pip install -q "ultralytics==8.4.174" "sahi==0.12.8" pycocotools ensemble-boxes pytest "git+https://github.com/ultralytics/CLIP.git"
Path("dronekit_src/dronekit").mkdir(parents=True, exist_ok=True)
Path("dronekit_src/tests").mkdir(parents=True, exist_ok=True)

## dronekit package (generated from the repo by `tools/build_phase0_notebook.py`; do not edit here)

In [ ]:
%%writefile dronekit_src/dronekit/__init__.py
"""dronekit: shared data loading, inference engine, scorer and plots for the drone-QA project.

One implementation of the evaluation protocol, imported by every notebook.
"""
__version__ = "0.1.0"

VISDRONE_CLASSES = ["pedestrian", "people", "bicycle", "car", "van",
                    "truck", "tricycle", "awning-tricycle", "bus", "motor"]

In [ ]:
%%writefile dronekit_src/dronekit/data.py
"""Dataset loading into one common structure.

Every dataset becomes a `Split`: a list of `ImageRecord`s, each holding
  boxes   (N, 7) float32: class_id (0-based), x1, y1, x2, y2, occlusion, truncation
  ignore  (M, 4) float32: x1, y1, x2, y2 of ignore regions
  sequence: video-sequence id parsed from the filename (bootstrap / CV grouping unit)

VisDrone rules: category 0 or score 0 = ignore region; category 11
("others") dropped; categories 1-10 -> class ids 0-9. Boxes are kept exactly as annotated
(not clipped), because the eval-v2 scorer used the raw x, y, w, h.
"""
from __future__ import annotations

import io
import urllib.request
import zipfile
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
from PIL import Image

from . import VISDRONE_CLASSES

VISDRONE_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-{split}.zip"


@dataclass
class ImageRecord:
    key: str                 # filename stem, unique within a split
    path: Path
    width: int
    height: int
    sequence: str
    boxes: np.ndarray = field(default_factory=lambda: np.zeros((0, 7), np.float32))
    ignore: np.ndarray = field(default_factory=lambda: np.zeros((0, 4), np.float32))


@dataclass
class Split:
    dataset: str
    name: str
    class_names: list
    images: list             # list[ImageRecord], sorted by key; position = COCO image id

    def __len__(self):
        return len(self.images)

    def subset(self, n):
        """First n images (smoke tests). Image ids stay positional."""
        return Split(self.dataset, self.name, self.class_names, self.images[:n])

    @property
    def n_objects(self):
        return int(sum(len(r.boxes) for r in self.images))

    @property
    def sequences(self):
        return sorted({r.sequence for r in self.images})


def visdrone_sequence(stem: str) -> str:
    """'0000001_02999_d_0000005' -> '0000001'. VisDrone-DET frames from one video share the first field."""
    return stem.split("_")[0]


def parse_visdrone_txt(text: str):
    """Return (boxes (N,7), ignore (M,4)) from one VisDrone annotation file."""
    boxes, ignore = [], []
    for row in text.strip().splitlines():
        v = row.strip().rstrip(",").split(",")
        if len(v) < 6:
            continue
        x, y, w, h, score, cat = map(int, v[:6])
        trunc = int(v[6]) if len(v) > 6 else -1
        occ = int(v[7]) if len(v) > 7 else -1
        if w <= 0 or h <= 0 or cat == 11:
            continue
        if score == 0 or cat == 0:
            ignore.append([x, y, x + w, y + h])
        elif 1 <= cat <= 10:
            boxes.append([cat - 1, x, y, x + w, y + h, occ, trunc])
    return (np.array(boxes, np.float32).reshape(-1, 7), np.array(ignore, np.float32).reshape(-1, 4))


def load_visdrone(split_dir, split_name: str | None = None) -> Split:
    """Load a VisDrone2019-DET split folder (with images/ and annotations/)."""
    split_dir = Path(split_dir)
    recs = []
    for ann in sorted((split_dir / "annotations").glob("*.txt")):
        img = split_dir / "images" / f"{ann.stem}.jpg"
        with Image.open(img) as im:
            W, H = im.size
        boxes, ignore = parse_visdrone_txt(ann.read_text())
        recs.append(ImageRecord(ann.stem, img, W, H, visdrone_sequence(ann.stem), boxes, ignore))
    return Split("visdrone", split_name or split_dir.name, list(VISDRONE_CLASSES), recs)


def download_visdrone(root, split: str) -> Path:
    """Download + extract one split ('train', 'val', 'test-dev') into root. Returns the split folder."""
    root = Path(root)
    d = root / f"VisDrone2019-DET-{split}"
    if not (d / "annotations").exists():
        root.mkdir(parents=True, exist_ok=True)
        print("Downloading", d.name)
        with urllib.request.urlopen(VISDRONE_URL.format(split=split)) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(root)
    return d


def split_stats(split: Split) -> dict:
    """Object counts and size shares (non-ignored objects)."""
    b = np.concatenate([r.boxes for r in split.images]) if split.images else np.zeros((0, 7))
    area = (b[:, 3] - b[:, 1]) * (b[:, 4] - b[:, 2])
    n = max(len(b), 1)
    out = dict(images=len(split), sequences=len(split.sequences), objects=len(b),
               objects_per_image=len(b) / max(len(split), 1),
               tiny=float((area < 16 ** 2).sum() / n),
               small=float(((area >= 16 ** 2) & (area < 32 ** 2)).sum() / n),
               medium_plus=float((area >= 32 ** 2).sum() / n))
    for k, c in enumerate(split.class_names):
        out[f"share_{c}"] = float((b[:, 0] == k).sum() / n)
    return out

In [ ]:
%%writefile dronekit_src/dronekit/infer.py
"""Inference engine ("evaluation v2"), raw-detection cache, and merge setups.

Same logic as evaluation(1).ipynb: SAHI's slicer (`get_slice_bboxes`) and SAHI's own merge
functions (`batched_nms`, `batched_greedy_nmm`) called directly with every setting explicit,
so nothing can switch silently. Never call `get_sliced_prediction`.

Pass keys: "full@S" = whole image at imgsz S; "sT@S" = T x T slices, each resized to S.
Detections are (N, 6) float32 arrays: x1, y1, x2, y2, score, class_id (0-based).
Slice passes also return slice boxes (N, 4) so the boundary filter can run later on CPU.

Cache: one parquet per model x split holding every raw pass down to CONF (0.01), plus a
JSON sidecar with timings and settings. Thresholds and merges can change later without the GPU.
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

CONF = 0.01           # detections kept down to 0.01
MAX_DET = 1000        # per forward pass
OVERLAP = 0.25        # slice overlap
SLICE_BATCH = 16
EDGE_MARGIN = 2.0     # boundary filter: px from an interior slice edge

# Named setups. "passes" are combined for one model, then merged.
SETUPS = {
    "whole1536":   dict(passes=["full@1536"], method=None),
    "best_sliced": dict(passes=["s512@1280", "full@1536"], method="NMS", metric="IOU", thr=0.6, edge=True),
    "fast":        dict(passes=["s512@1024", "full@1536"], method="NMS", metric="IOU", thr=0.6, edge=True),
}
ENSEMBLE_WBF_THR = 0.65


def parse_pass(key: str):
    kind, sz = key.split("@")
    return ("full", None, int(sz)) if kind == "full" else ("slice", int(kind[1:]), int(sz))


def slice_boxes(H: int, W: int, T: int, overlap: float = OVERLAP):
    from sahi.slicing import get_slice_bboxes
    return get_slice_bboxes(image_height=H, image_width=W, slice_height=T, slice_width=T,
                            auto_slice_resolution=False, overlap_height_ratio=overlap, overlap_width_ratio=overlap)


# --------------------------------------------------------------------------- running a model
def _arr(r, cls_map=None):
    b = r.boxes
    if b is None or len(b) == 0:
        return np.zeros((0, 6), np.float32)
    a = np.c_[b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()].astype(np.float32)
    if cls_map is not None:          # model class index -> dataset class id
        a[:, 5] = np.vectorize(cls_map.__getitem__)(a[:, 5].astype(int))
    return a


def run_pass_on_image(model, img, key, *, conf=CONF, max_det=MAX_DET, half=True, device=0,
                      slice_batch=SLICE_BATCH, overlap=OVERLAP, cls_map=None):
    """One pass on one BGR image. Returns (dets (N,6), slice_boxes (N,4) or None)."""
    kind, T, S = parse_pass(key)
    kw = dict(imgsz=S, conf=conf, max_det=max_det, half=half, device=device, verbose=False)
    if kind == "full":
        return _arr(model.predict(img, **kw)[0], cls_map), None
    H, W = img.shape[:2]
    boxes = slice_boxes(H, W, T, overlap)
    crops = [np.ascontiguousarray(img[y1:y2, x1:x2]) for x1, y1, x2, y2 in boxes]
    D, SB = [], []
    for i in range(0, len(crops), slice_batch):
        rs = model.predict(crops[i:i + slice_batch], **kw)
        for r, (x1, y1, x2, y2) in zip(rs, boxes[i:i + slice_batch]):
            a = _arr(r, cls_map)
            a[:, [0, 2]] += x1
            a[:, [1, 3]] += y1
            D.append(a)
            SB.append(np.tile(np.float32([x1, y1, x2, y2]), (len(a), 1)))
    return np.concatenate(D), np.concatenate(SB).reshape(-1, 4)


COLS = ["image", "pass", "x1", "y1", "x2", "y2", "score", "cls", "sx1", "sy1", "sx2", "sy2"]


def run_and_cache(model, split, passes, out_path, *, meta=None, cuda=True, **kw):
    """Run every pass on every image of `split`; write <out_path>.parquet + <out_path>.json.

    Skips work if the parquet already exists. Returns the cache path.
    Latency = GPU time per image per pass (torch.cuda.synchronize around each image).
    """
    import cv2
    out_path = Path(out_path)
    pq, js = out_path.with_suffix(".parquet"), out_path.with_suffix(".json")
    if pq.exists() and js.exists():
        return pq
    sync = (lambda: __import__("torch").cuda.synchronize()) if cuda else (lambda: None)
    frames, ms = [], {}
    for key in passes:
        t = 0.0
        for rec in split.images:
            img = cv2.imread(str(rec.path))
            sync(); t0 = time.perf_counter()
            d, sb = run_pass_on_image(model, img, key, **kw)
            sync(); t += time.perf_counter() - t0
            if sb is None:
                sb = np.full((len(d), 4), np.nan, np.float32)
            f = pd.DataFrame(np.c_[d, sb].astype(np.float32), columns=COLS[2:])
            f.insert(0, "pass", key); f.insert(0, "image", rec.key)
            frames.append(f)
        ms[key] = 1000 * t / max(len(split), 1)
        print(f"  {out_path.name} {key}: {ms[key]:.0f} ms/img")
    df = pd.concat(frames, ignore_index=True)
    df["image"] = df["image"].astype("category"); df["pass"] = df["pass"].astype("category")
    df["cls"] = df["cls"].astype(np.int16)
    pq.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(pq, compression="zstd", index=False)
    js.write_text(json.dumps(dict(meta or {}, split=split.name, dataset=split.dataset, n_images=len(split),
                                  passes=list(passes), ms_per_img=ms, conf=kw.get("conf", CONF),
                                  max_det=kw.get("max_det", MAX_DET), overlap=kw.get("overlap", OVERLAP),
                                  rows=len(df)), indent=2))
    return pq


def load_cache(path):
    """-> (raw, meta). raw[pass][image_key] = (dets (N,6), slice_boxes (N,4) or None)."""
    path = Path(path)
    df = pd.read_parquet(path.with_suffix(".parquet"))
    meta = json.loads(path.with_suffix(".json").read_text())
    raw = {p: {} for p in meta.get("passes", [])}   # a pass with zero detections still exists
    for (p, im), g in df.groupby(["pass", "image"], observed=True, sort=False):
        a = g[COLS[2:]].to_numpy(np.float32)
        raw.setdefault(str(p), {})[str(im)] = (a[:, :6], None if parse_pass(str(p))[0] == "full" else a[:, 6:])
    return raw, meta


# --------------------------------------------------------------------------- merging
def edge_filter(d, sb, W, H, margin=EDGE_MARGIN):
    """Drop slice detections within `margin` px of an *interior* slice edge (a cut object).
    Edges lying on the image border are not interior and never filter anything."""
    if sb is None or len(d) == 0:
        return d
    x1, y1, x2, y2 = d[:, 0], d[:, 1], d[:, 2], d[:, 3]
    sx1, sy1, sx2, sy2 = sb[:, 0], sb[:, 1], sb[:, 2], sb[:, 3]
    cut = (((sx1 > 0) & (x1 <= sx1 + margin)) | ((sy1 > 0) & (y1 <= sy1 + margin)) |
           ((sx2 < W) & (x2 >= sx2 - margin)) | ((sy2 < H) & (y2 >= sy2 - margin)))
    return d[~cut]


def _overlap(a, b, metric):
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    aa = (a[2] - a[0]) * (a[3] - a[1]); ab = (b[2] - b[0]) * (b[3] - b[1])
    den = min(aa, ab) if metric == "IOS" else (aa + ab - inter)
    return inter / den if den > 0 else 0.0


def merge(parts, method="NMS", metric="IOU", thr=0.5):
    """Class-wise merge with SAHI's own functions."""
    from sahi.postprocess.combine import batched_greedy_nmm, batched_nms
    parts = [p for p in parts if len(p)]
    if not parts:
        return np.zeros((0, 6), np.float32)
    P = np.concatenate(parts)
    if method == "NMS":
        return P[batched_nms(P, match_metric=metric, match_threshold=thr)]
    if method == "GREEDYNMM":                     # same logic as SAHI's _apply_merge: union box, max score
        out = []
        for keep, mlist in batched_greedy_nmm(P, match_metric=metric, match_threshold=thr).items():
            b = P[keep].copy()
            for j in mlist:
                if _overlap(b, P[j], metric) >= thr:
                    b[0], b[1] = min(b[0], P[j, 0]), min(b[1], P[j, 1])
                    b[2], b[3] = max(b[2], P[j, 2]), max(b[3], P[j, 3]); b[4] = max(b[4], P[j, 4])
            out.append(b)
        return np.array(out, np.float32).reshape(-1, 6)
    raise ValueError(method)


def wbf(dets_per_model, W, H, thr=ENSEMBLE_WBF_THR, skip=CONF):
    """Weighted box fusion across models (each model already merged)."""
    from ensemble_boxes import weighted_boxes_fusion
    bl, sl, ll = [], [], []
    for d in dets_per_model:
        bl.append(np.clip(d[:, :4] / [W, H, W, H], 0, 1).tolist()); sl.append(d[:, 4].tolist()); ll.append(d[:, 5].tolist())
    b, s, l = weighted_boxes_fusion(bl, sl, ll, iou_thr=thr, skip_box_thr=skip)
    return np.c_[np.asarray(b).reshape(-1, 4) * [W, H, W, H], s, l].astype(np.float32).reshape(-1, 6)


def apply_setup(raw, split, setup):
    """Merge cached raw passes into final detections. Returns (dets {image_key: (N,6)}, merge ms/img)."""
    if isinstance(setup, str):
        setup = SETUPS[setup]
    passes = setup["passes"]
    missing = [p for p in passes if p not in raw]
    if missing:
        raise KeyError(f"passes not in cache: {missing}")
    out, t0 = {}, time.perf_counter()
    single_full = len(passes) == 1 and parse_pass(passes[0])[0] == "full"
    for rec in split.images:
        parts = []
        for k in passes:
            d, sb = raw[k].get(rec.key, (np.zeros((0, 6), np.float32), None))
            parts.append(edge_filter(d, sb, rec.width, rec.height) if setup.get("edge") else d)
        out[rec.key] = parts[0] if single_full else merge(parts, setup["method"], setup["metric"], setup["thr"])
    return out, 1000 * (time.perf_counter() - t0) / max(len(split), 1)


def ensemble(dets_list, split, thr=ENSEMBLE_WBF_THR):
    """WBF of several models' merged detections, per image."""
    return {r.key: wbf([d[r.key] for d in dets_list], r.width, r.height, thr) for r in split.images}


def setup_ms(meta, setup, merge_ms=0.0):
    """Latency of a setup = sum of its passes' GPU ms/img + merge time (as in eval v2)."""
    if isinstance(setup, str):
        setup = SETUPS[setup]
    return sum(meta["ms_per_img"][p] for p in setup["passes"]) + merge_ms

In [ ]:
%%writefile dronekit_src/dronekit/score.py
"""Scorer ("evaluation v2"): COCO AP via pycocotools with the VisDrone settings.

- maxDets = [1, 100, 500]; every number reads the 500 index (`stats` hard-codes 100)
- ignore regions inserted as crowd boxes for every class -> detections inside are neither TP nor FP
- "others" already dropped by the loader
- size buckets: tiny < 16^2, small 16^2-32^2, medium+ >= 32^2 (plus COCO small < 32^2)
Also: per-class AP50, precision/recall at a fixed confidence, PR curves, confusion matrix.
"""
from __future__ import annotations

import contextlib
import io

import numpy as np

AREAS = {"all": [0, 1e10], "tiny": [0, 16 ** 2], "small": [16 ** 2, 32 ** 2],
         "coco_small": [0, 32 ** 2], "medium+": [32 ** 2, 1e10]}
MAX_DETS = [1, 100, 500]


def coco_gt(split) -> dict:
    """COCO-format GT dict. Image id = position in split.images; category id = class_id + 1."""
    images, anns, aid = [], [], 1
    nc = len(split.class_names)
    for i, r in enumerate(split.images):
        images.append(dict(id=i, file_name=r.path.name, width=r.width, height=r.height))
        for x1, y1, x2, y2 in r.ignore:
            w, h = float(x2 - x1), float(y2 - y1)
            for c in range(1, nc + 1):
                anns.append(dict(id=aid, image_id=i, category_id=c, bbox=[float(x1), float(y1), w, h],
                                 area=w * h, iscrowd=1)); aid += 1
        for c, x1, y1, x2, y2, *_ in r.boxes:
            w, h = float(x2 - x1), float(y2 - y1)
            anns.append(dict(id=aid, image_id=i, category_id=int(c) + 1, bbox=[float(x1), float(y1), w, h],
                             area=w * h, iscrowd=0)); aid += 1
    return dict(images=images, annotations=anns,
                categories=[dict(id=k + 1, name=n) for k, n in enumerate(split.class_names)])


def _coco(split):
    from pycocotools.coco import COCO
    with contextlib.redirect_stdout(io.StringIO()):
        G = COCO(); G.dataset = coco_gt(split); G.createIndex()
    return G


def _ap(p):
    p = p[p > -1]
    return float(p.mean()) if p.size else float("nan")


def run_cocoeval(split, dets, gt=None):
    """dets: {image_key: (N,6) x1,y1,x2,y2,score,cls}. Returns the evaluated+accumulated COCOeval, or None."""
    from pycocotools.cocoeval import COCOeval
    G = gt or _coco(split)
    rows = []
    for i, r in enumerate(split.images):
        d = dets.get(r.key)
        if d is not None and len(d):
            rows.append(np.c_[np.full(len(d), i), d[:, 0], d[:, 1], d[:, 2] - d[:, 0], d[:, 3] - d[:, 1],
                              d[:, 4], d[:, 5] + 1])
    if not rows:
        return None
    with contextlib.redirect_stdout(io.StringIO()):
        dt = G.loadRes(np.concatenate(rows).astype(np.float64))
        E = COCOeval(G, dt, "bbox")
        E.params.imgIds = list(range(len(split.images)))
        E.params.maxDets = list(MAX_DETS)
        E.params.areaRng = list(AREAS.values()); E.params.areaRngLbl = list(AREAS.keys())
        E.evaluate(); E.accumulate()
    return E


def metrics_from_eval(E, class_names):
    """-> (metrics dict, per-class AP50 dict). Same keys as evaluation v2."""
    if E is None:
        return {"AP": 0.0, "AP50": 0.0}, {c: 0.0 for c in class_names}
    P = E.eval["precision"]                       # [IoU, recall, class, area, maxDets]
    m = {"AP": _ap(P[:, :, :, 0, -1]), "AP50": _ap(P[0, :, :, 0, -1]), "AP75": _ap(P[5, :, :, 0, -1])}
    for a, lbl in enumerate(AREAS):
        if lbl != "all":
            m[f"AP_{lbl}"] = _ap(P[:, :, :, a, -1]); m[f"AP50_{lbl}"] = _ap(P[0, :, :, a, -1])
    per_cls = {c: _ap(P[0, :, k, 0, -1]) for k, c in enumerate(class_names)}
    return m, per_cls


def evaluate(split, dets, gt=None):
    """-> (metrics, per-class AP50, COCOeval)."""
    E = run_cocoeval(split, dets, gt)
    m, pc = metrics_from_eval(E, split.class_names)
    return m, pc, E


def matches_at_iou50(E):
    """Per-detection outcome at IoU 0.5, area 'all', maxDets 500, using COCOeval's own matching.

    Returns (scores, is_tp, n_gt) per class: scores/is_tp are arrays of non-ignored detections
    (detections matched to a crowd/ignore box are dropped: neither TP nor FP).
    """
    p = E.params
    nA, nI = len(p.areaRng), len(p.imgIds)
    out = {}
    for k, cat in enumerate(p.catIds):
        sc, tp, ngt = [], [], 0
        for i in range(nI):
            e = E.evalImgs[k * nA * nI + 0 * nI + i]           # area index 0 = 'all'
            if e is None:
                continue
            ngt += int((~np.asarray(e["gtIgnore"], bool)).sum())
            dm, di = e["dtMatches"][0], e["dtIgnore"][0]
            keep = ~np.asarray(di, bool)
            sc.append(np.asarray(e["dtScores"])[keep]); tp.append((np.asarray(dm) > 0)[keep])
        out[cat] = (np.concatenate(sc) if sc else np.zeros(0), np.concatenate(tp) if tp else np.zeros(0, bool), ngt)
    return out


def pr_at_conf(E, conf):
    """Pooled (all classes) precision/recall/F1 at IoU 0.5 for detections with score >= conf."""
    if E is None:
        return dict(precision=0.0, recall=0.0, f1=0.0, tp=0, fp=0, n_gt=0)
    tp = fp = ngt = 0
    for sc, t, n in matches_at_iou50(E).values():
        s = sc >= conf
        tp += int(t[s].sum()); fp += int((~t[s]).sum()); ngt += n
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / ngt if ngt else 0.0
    return dict(precision=p, recall=r, f1=2 * p * r / (p + r) if p + r else 0.0, tp=tp, fp=fp, n_gt=ngt)


def best_f1_conf(E, grid=None):
    """Confidence with the highest pooled F1 (pick on val, reuse unchanged on test)."""
    grid = np.round(np.arange(0.05, 0.951, 0.01), 2) if grid is None else grid
    M = matches_at_iou50(E)
    sc = np.concatenate([v[0] for v in M.values()]); t = np.concatenate([v[1] for v in M.values()])
    ngt = sum(v[2] for v in M.values())
    best = (0.0, float(grid[0]))
    for c in grid:
        s = sc >= c; tp = t[s].sum(); fp = (~t[s]).sum()
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / ngt if ngt else 0.0
        f1 = 2 * p * r / (p + r) if p + r else 0.0
        if f1 > best[0]:
            best = (f1, float(c))
    return best[1]


def pr_curves(E):
    """-> (recall thresholds (101,), precision (101, n_classes)) at IoU 0.5, area all, maxDets 500.
    COCO's interpolated precision; -1 (no GT) becomes NaN."""
    P = E.eval["precision"][0, :, :, 0, -1].astype(float)
    P[P < 0] = np.nan
    return np.asarray(E.params.recThrs), P


def _iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter, inter / np.clip(aa[:, None] + ab[None, :] - inter, 1e-9, None), aa


def confusion_matrix(split, dets, conf, iou_thr=0.5):
    """(nc+1, nc+1) counts; rows = true class (+ last row 'background' = false positives),
    columns = predicted class (+ last column 'background' = missed objects).

    Class-agnostic greedy matching by score at IoU >= iou_thr. Unmatched detections whose area
    is >= 50% inside an ignore region are dropped, mirroring the crowd rule (COCO uses
    intersection / detection area for crowd boxes).
    """
    nc = len(split.class_names)
    cm = np.zeros((nc + 1, nc + 1), np.int64)
    for r in split.images:
        d = dets.get(r.key, np.zeros((0, 6), np.float32))
        d = d[d[:, 4] >= conf]
        d = d[np.argsort(-d[:, 4], kind="stable")]
        g = r.boxes
        used = np.zeros(len(g), bool)
        unmatched = []
        if len(d) and len(g):
            _, iou, _ = _iou(d[:, :4], g[:, 1:5])
        for j in range(len(d)):
            if len(g):
                cand = np.where(~used & (iou[j] >= iou_thr))[0]
                if len(cand):
                    gi = cand[np.argmax(iou[j, cand])]
                    used[gi] = True
                    cm[int(g[gi, 0]), int(d[j, 5])] += 1
                    continue
            unmatched.append(j)
        if unmatched:
            u = d[unmatched]
            if len(r.ignore):
                inter, _, area = _iou(u[:, :4], r.ignore)
                u = u[(inter / np.clip(area[:, None], 1e-9, None)).max(1) < 0.5]
            for c in u[:, 5].astype(int):
                cm[nc, c] += 1
        for gi in np.where(~used)[0]:
            cm[int(g[gi, 0]), nc] += 1
    return cm


def normalize_rows(cm):
    s = cm.sum(1, keepdims=True).astype(float)
    return np.divide(cm, s, out=np.zeros_like(cm, float), where=s > 0)

In [ ]:
%%writefile dronekit_src/dronekit/plots.py
"""Figures: PR curves, normalized confusion matrix, per-class bars, training curves.

Paper figures (white background). Colors follow a validated categorical order; one-series
panels use slot 1, val vs test-dev comparisons use slots 1 and 2 with a legend.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SEQ_CMAP = ["#ffffff", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"


def _plt():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK, "text.color": INK,
                         "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
                         "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
                         "legend.frameon": False, "savefig.dpi": 200, "savefig.bbox": "tight",
                         "figure.facecolor": "white", "axes.facecolor": "white"})
    return plt


def _save(fig, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path)
    import matplotlib.pyplot as plt
    plt.close(fig)
    return path


def plot_pr_curves(curves, class_names, path, title=""):
    """curves: {label: (recall (R,), precision (R, nc))}, e.g. {"val": ..., "test-dev": ...}.
    Small multiples, one panel per class (10 classes would exceed the 8 categorical hues)."""
    plt = _plt()
    nc = len(class_names); ncol = 5; nrow = int(np.ceil(nc / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.3 * ncol, 2.2 * nrow), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    for k, c in enumerate(class_names):
        ax = axes[k]
        for j, (lbl, (rec, prec)) in enumerate(curves.items()):
            p = prec[:, k]
            ap = np.nanmean(p) if np.isfinite(p).any() else float("nan")
            ax.plot(rec, p, color=SERIES[j], lw=1.6, label=f"{lbl} (AP50 {ap:.2f})")
        ax.set_title(c, fontsize=9); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        ax.legend(fontsize=6.5, loc="lower left")
    for ax in axes[nc:]:
        ax.axis("off")
    for ax in axes[(nrow - 1) * ncol:]:
        ax.set_xlabel("recall")
    for ax in axes[::ncol]:
        ax.set_ylabel("precision")
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    return _save(fig, path)


def plot_confusion(cm_norm, class_names, path, title="", conf=None):
    """Row-normalized confusion matrix (rows = true class + 'background' FP row)."""
    plt = _plt()
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("seq_blue", SEQ_CMAP)
    labels = list(class_names) + ["background"]
    n = len(labels)
    fig, ax = plt.subplots(figsize=(0.62 * n + 1.6, 0.55 * n + 1.2))
    ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=1)
    ax.grid(False)
    for i in range(n):
        for j in range(n):
            v = cm_norm[i, j]
            if v >= 0.005:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if v > 0.55 else INK)
    ax.set_xticks(range(n), labels, rotation=45, ha="right"); ax.set_yticks(range(n), labels)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    sub = f" (conf ≥ {conf:.2f}, IoU ≥ 0.5)" if conf is not None else ""
    ax.set_title(title + sub, fontsize=9)
    fig.tight_layout()
    return _save(fig, path)


def plot_per_class_bars(df, path, title="", cols=("val", "test-dev")):
    """df: index = class names, columns include `cols` (AP50 values). Grouped horizontal bars."""
    plt = _plt()
    n = len(df); h = 0.38
    fig, ax = plt.subplots(figsize=(6.2, 0.42 * n + 1.0))
    y = np.arange(n)
    for j, c in enumerate(cols):
        if c not in df:
            continue
        ax.barh(y + (j - (len(cols) - 1) / 2) * (h + 0.02), df[c].values, height=h, color=SERIES[j], label=c)
    ax.set_yticks(y, df.index); ax.invert_yaxis()
    ax.set_xlim(0, 1); ax.set_xlabel("AP50"); ax.grid(axis="y", visible=False)
    ax.legend(loc="lower right", ncol=len(cols))
    ax.set_title(title, fontsize=10)
    fig.tight_layout()
    return _save(fig, path)


def plot_training_curves(results_csv, path, title=""):
    """Ultralytics results.csv -> small multiples: each loss and each validator metric in its own panel.
    Note: these metrics come from the Ultralytics validator, not the eval-v2 scorer."""
    import pandas as pd
    plt = _plt()
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    x = df["epoch"] if "epoch" in df else np.arange(1, len(df) + 1)
    cols = [c for c in df.columns if c.startswith(("train/", "val/", "metrics/"))]
    ncol = 4; nrow = int(np.ceil(len(cols) / ncol)) or 1
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.0 * nrow), sharex=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, cols):
        ax.plot(x, df[c], color=SERIES[0], lw=1.6, marker="o", ms=3)
        ax.set_title(c.replace("metrics/", "").replace("(B)", ""), fontsize=8.5)
    for ax in axes[len(cols):]:
        ax.axis("off")
    for ax in axes[-ncol:]:
        ax.set_xlabel("epoch")
    fig.suptitle(title + "  (validator metrics: Ultralytics, imgsz from training)", fontsize=9.5)
    fig.tight_layout()
    return _save(fig, path)

In [ ]:
%%writefile dronekit_src/dronekit/report.py
"""Training provenance from Ultralytics run files, and small markdown-table helpers."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd


def read_results_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df


def read_run(weights=None, results_csv=None, args_yaml=None):
    """Collect what a training run recorded: checkpoint train_args, args.yaml, results.csv.
    Every value comes from those files; missing ones stay None."""
    info, ta = {}, {}
    if weights is not None and Path(weights).exists():
        try:
            import torch
            ck = torch.load(weights, map_location="cpu", weights_only=False)
            ta = dict(ck.get("train_args") or {})
            info.update(ultralytics_version=ck.get("version"), checkpoint_date=ck.get("date"))
        except Exception as e:                       # keep going; provenance is best-effort per field
            info["checkpoint_error"] = repr(e)[:200]
    if args_yaml is not None and Path(args_yaml).exists():
        import yaml
        ta = {**ta, **(yaml.safe_load(Path(args_yaml).read_text()) or {})}
    for k in ("model", "data", "imgsz", "batch", "optimizer", "lr0", "epochs", "time", "seed", "device"):
        info[f"arg_{k}"] = ta.get(k)
    if results_csv is not None and Path(results_csv).exists():
        df = read_results_csv(results_csv)
        info["epochs_completed"] = int(len(df))
        if "time" in df:
            info["train_hours"] = round(float(df["time"].iloc[-1]) / 3600, 2)
        key = "metrics/mAP50-95(B)"
        if key in df:
            b = int(df[key].idxmax())
            info["best_epoch"] = int(df["epoch"].iloc[b]) if "epoch" in df else b + 1
            info["best_val_mAP50-95_ultralytics"] = round(float(df[key].iloc[b]), 4)
            if "metrics/mAP50(B)" in df:
                info["best_val_mAP50_ultralytics"] = round(float(df["metrics/mAP50(B)"].iloc[b]), 4)
    return info


def md_table(df, floatfmt="{:.3f}"):
    """DataFrame -> GitHub markdown table (no tabulate dependency)."""
    def f(v):
        if isinstance(v, (float, np.floating)):
            return "" if np.isnan(v) else floatfmt.format(v)
        return "" if v is None else str(v)
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(f(r[c]) for c in cols) + " |")
    return "\n".join(lines)

In [ ]:
%%writefile dronekit_src/tests/conftest.py
import sys
from pathlib import Path

import numpy as np
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from dronekit.data import ImageRecord, Split  # noqa: E402


def make_split(images, class_names=("a", "b")):
    """images: list of dict(boxes=[[cls,x1,y1,x2,y2],...], ignore=[[x1,y1,x2,y2],...], wh=(W,H))."""
    recs = []
    for i, im in enumerate(images):
        b = np.array([list(r) + [0, 0] for r in im.get("boxes", [])], np.float32).reshape(-1, 7)
        g = np.array(im.get("ignore", []), np.float32).reshape(-1, 4)
        W, H = im.get("wh", (1000, 1000))
        recs.append(ImageRecord(f"img{i:03d}", Path(f"img{i:03d}.jpg"), W, H, "seq0", b, g))
    return Split("toy", "toy", list(class_names), recs)


@pytest.fixture
def split_factory():
    return make_split

In [ ]:
%%writefile dronekit_src/tests/test_data.py
import numpy as np

from dronekit import data


def test_parse_visdrone_rules():
    txt = "\n".join([
        "10,20,30,40,1,4,0,1",     # car (cat 4 -> class 3), trunc 0, occ 1
        "0,0,50,50,0,1,0,0",       # score 0 -> ignore region
        "5,5,10,10,1,0,0,0",       # category 0 -> ignore region
        "1,1,5,5,1,11,0,0",        # others -> dropped
        "1,1,0,5,1,3,0,0",         # zero width -> dropped
        "7,8,2,3,1,10,1,2,",       # trailing comma; motor (class 9)
    ])
    boxes, ign = data.parse_visdrone_txt(txt)
    assert boxes.tolist() == [[3, 10, 20, 40, 60, 1, 0], [9, 7, 8, 9, 11, 2, 1]]
    assert ign.tolist() == [[0, 0, 50, 50], [5, 5, 15, 15]]


def test_sequence_id():
    assert data.visdrone_sequence("0000001_02999_d_0000005") == "0000001"
    assert data.visdrone_sequence("9999938_00000_d_0000207") == "9999938"


def test_split_stats(split_factory):
    sp = split_factory([dict(boxes=[[0, 0, 0, 10, 10], [1, 0, 0, 40, 40]])])
    st = data.split_stats(sp)
    assert st["objects"] == 2 and np.isclose(st["tiny"], 0.5) and np.isclose(st["medium_plus"], 0.5)

In [ ]:
%%writefile dronekit_src/tests/test_infer.py
import numpy as np
import pytest

from dronekit import infer


def test_box_cut_by_interior_slice_edge_is_removed():
    W, H = 1000, 1000
    sb = np.float32([0, 0, 600, 600])                    # slice whose right and bottom edges are interior
    d = np.float32([[550, 100, 600, 150, .9, 0],          # touches interior right edge -> cut object
                    [100, 560, 150, 599, .9, 0],          # within 2 px of interior bottom edge -> cut
                    [100, 100, 150, 150, .9, 0],          # well inside -> kept
                    [0, 300, 40, 340, .9, 0]])            # touches the image border (x=0), not interior -> kept
    out = infer.edge_filter(d, np.tile(sb, (len(d), 1)), W, H)
    assert out[:, 0].tolist() == [100, 0]


def test_edge_on_image_border_is_not_interior():
    W, H = 1000, 1000
    sb = np.float32([400, 400, 1000, 1000])               # right/bottom edges are the image border
    d = np.float32([[900, 900, 1000, 1000, .9, 0],        # touches border -> kept
                    [400, 500, 450, 550, .9, 0]])         # touches interior left edge -> removed
    out = infer.edge_filter(d, np.tile(sb, (len(d), 1)), W, H)
    assert out[:, :4].tolist() == [[900, 900, 1000, 1000]]


def test_full_pass_is_never_edge_filtered():
    d = np.float32([[0, 0, 10, 10, .9, 0]])
    assert infer.edge_filter(d, None, 100, 100) is d


def test_classwise_nms():
    d1 = np.float32([[0, 0, 100, 100, .9, 0], [0, 0, 100, 100, .8, 1]])
    d2 = np.float32([[2, 2, 100, 100, .7, 0]])            # duplicate of the class-0 box
    out = infer.merge([d1, d2], "NMS", "IOU", 0.6)
    assert sorted(out[:, 4].tolist()) == pytest.approx([0.8, 0.9])


def test_apply_setup_and_cache_roundtrip(tmp_path, split_factory):
    sp = split_factory([dict(wh=(1000, 1000)), dict(wh=(1000, 1000))])
    raw = {"full@1536": {"img000": (np.float32([[10, 10, 50, 50, .9, 0]]), None),
                         "img001": (np.zeros((0, 6), np.float32), None)},
           "s512@1280": {"img000": (np.float32([[11, 11, 50, 50, .8, 0], [480, 10, 512, 40, .9, 1]]),
                                    np.float32([[0, 0, 512, 512], [0, 0, 512, 512]])),
                         "img001": (np.float32([[100, 100, 130, 130, .5, 1]]), np.float32([[0, 0, 512, 512]]))}}
    dets, _ = infer.apply_setup(raw, sp, "best_sliced")
    # img000: slice dup of the full box removed by NMS; box at x2=512 cut by interior edge -> dropped
    assert dets["img000"].tolist() == [[10, 10, 50, 50, pytest.approx(.9), 0]]
    assert len(dets["img001"]) == 1
    whole, _ = infer.apply_setup(raw, sp, "whole1536")
    assert len(whole["img000"]) == 1

    # cache roundtrip through parquet
    import json
    import pandas as pd
    rows = []
    for p, per in raw.items():
        for k, (d, sb) in per.items():
            sb = np.full((len(d), 4), np.nan, np.float32) if sb is None else sb
            for r in np.c_[d, sb]:
                rows.append([k, p, *r])
    df = pd.DataFrame(rows, columns=infer.COLS)
    df.to_parquet(tmp_path / "c.parquet")
    (tmp_path / "c.json").write_text(json.dumps({"ms_per_img": {"full@1536": 30.0, "s512@1280": 400.0}}))
    raw2, meta = infer.load_cache(tmp_path / "c")
    d2, _ = infer.apply_setup(raw2, sp, "best_sliced")
    assert np.allclose(d2["img000"], dets["img000"])
    assert infer.setup_ms(meta, "best_sliced", 1.0) == pytest.approx(431.0)


def test_cache_with_zero_detections_keeps_its_passes(tmp_path, split_factory):
    import json
    import pandas as pd
    sp = split_factory([dict()])
    pd.DataFrame(columns=infer.COLS).astype({c: "float32" for c in infer.COLS[2:]}).to_parquet(tmp_path / "e.parquet")
    (tmp_path / "e.json").write_text(json.dumps({"passes": ["s512@1280", "full@1536"],
                                                 "ms_per_img": {"s512@1280": 1.0, "full@1536": 1.0}}))
    raw, _ = infer.load_cache(tmp_path / "e")
    dets, _ = infer.apply_setup(raw, sp, "best_sliced")
    assert len(dets["img000"]) == 0


def test_wbf_fuses_matching_boxes():
    a = np.float32([[100, 100, 200, 200, .9, 0]])
    b = np.float32([[102, 102, 202, 202, .7, 0]])
    out = infer.wbf([a, b], 1000, 1000, thr=0.65)
    assert len(out) == 1 and 100 <= out[0, 0] <= 102

In [ ]:
%%writefile dronekit_src/tests/test_score.py
import numpy as np
import pytest

from dronekit import score


def dets(rows):
    return np.array(rows, np.float32).reshape(-1, 6)


def test_box_inside_ignore_region_is_neither_tp_nor_fp(split_factory):
    sp = split_factory([dict(boxes=[[0, 100, 100, 200, 200]], ignore=[[500, 500, 700, 700]])])
    d = {"img000": dets([[100, 100, 200, 200, 0.9, 0],      # true positive
                         [550, 550, 600, 600, 0.95, 0]])}   # fully inside the ignore region, highest score
    m, pc, E = score.evaluate(sp, d)
    assert m["AP50"] == pytest.approx(1.0)                  # an FP ranked first would cut AP below 1
    pr = score.pr_at_conf(E, 0.0)
    assert (pr["tp"], pr["fp"], pr["n_gt"]) == (1, 0, 1)    # ignored det counted as neither
    cm = score.confusion_matrix(sp, d, conf=0.0)
    assert cm[-1].sum() == 0                                # no background (FP) entry either


def test_same_box_outside_ignore_region_is_fp(split_factory):
    # control: the identical detection without an ignore region does count as a false positive
    sp = split_factory([dict(boxes=[[0, 100, 100, 200, 200]])])
    d = {"img000": dets([[100, 100, 200, 200, 0.9, 0], [550, 550, 600, 600, 0.95, 0]])}
    m, _, E = score.evaluate(sp, d)
    assert m["AP50"] < 1.0
    assert score.pr_at_conf(E, 0.0)["fp"] == 1


def test_hand_computed_ap_three_boxes(split_factory):
    # 2 GT, 3 detections ranked: TP (0.9), FP (0.8), TP (0.7). Exact boxes -> same at every IoU.
    # precision = 1, 1/2, 2/3 at recall = 1/2, 1/2, 1. COCO envelope: 1 for r <= 0.5 (51 of the
    # 101 recall points), 2/3 for r in 0.51..1.00 (50 points).  AP = (51 + 50*2/3) / 101.
    sp = split_factory([dict(boxes=[[0, 100, 100, 200, 200], [0, 400, 400, 500, 500]])])
    d = {"img000": dets([[100, 100, 200, 200, 0.9, 0],
                         [700, 700, 800, 800, 0.8, 0],
                         [400, 400, 500, 500, 0.7, 0]])}
    m, pc, E = score.evaluate(sp, d)
    expected = (51 + 50 * 2 / 3) / 101
    assert pc["a"] == pytest.approx(expected, abs=1e-6)
    assert m["AP50"] == pytest.approx(expected, abs=1e-6)   # class "b" has no GT -> excluded
    assert m["AP"] == pytest.approx(expected, abs=1e-6)
    assert np.isnan(pc["b"])
    pr = score.pr_at_conf(E, 0.75)                          # keeps TP 0.9 and FP 0.8
    assert pr["precision"] == pytest.approx(0.5) and pr["recall"] == pytest.approx(0.5)


def test_size_buckets(split_factory):
    # one tiny (10x10), one small (20x20), one medium+ (40x40) object, all detected exactly
    sp = split_factory([dict(boxes=[[0, 0, 0, 10, 10], [0, 100, 100, 120, 120], [0, 300, 300, 340, 340]])])
    d = {"img000": dets([[0, 0, 10, 10, .9, 0], [100, 100, 120, 120, .8, 0], [300, 300, 340, 340, .7, 0]])}
    m, _, _ = score.evaluate(sp, d)
    for k in ("AP50_tiny", "AP50_small", "AP50_medium+", "AP50"):
        assert m[k] == pytest.approx(1.0), k


def test_max_dets_500_is_used(split_factory):
    # 200 GT objects, all detected: with maxDets 100 recall would cap at 0.5
    boxes = [[0, 20 * (i % 20), 20 * (i // 20) * 2, 20 * (i % 20) + 15, 20 * (i // 20) * 2 + 15] for i in range(200)]
    sp = split_factory([dict(boxes=boxes, wh=(1000, 1000))])
    d = {"img000": dets([[b[1], b[2], b[3], b[4], 0.5 + i / 1000, 0] for i, b in enumerate(boxes)])}
    m, _, E = score.evaluate(sp, d)
    assert m["AP50"] == pytest.approx(1.0)
    assert score.pr_at_conf(E, 0.0)["recall"] == pytest.approx(1.0)


def test_confusion_matrix_counts(split_factory):
    sp = split_factory([dict(boxes=[[0, 0, 0, 50, 50], [1, 100, 100, 150, 150], [1, 300, 300, 350, 350]])])
    d = {"img000": dets([[0, 0, 50, 50, .9, 0],            # a -> a
                         [100, 100, 150, 150, .8, 0],      # b predicted as a
                         [600, 600, 650, 650, .7, 1]])}    # background FP as b; GT at 300 missed
    cm = score.confusion_matrix(sp, d, conf=0.5)
    assert cm.tolist() == [[1, 0, 0], [1, 0, 1], [0, 1, 0]]
    assert np.allclose(score.normalize_rows(cm)[1], [0.5, 0, 0.5])

In [ ]:
# --- Unit tests (CPU, seconds) ----------------------------------------------
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "dronekit_src/tests"],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed: stop here and send the log"

In [ ]:
# --- Config ----------------------------------------------------------------
import os, sys, gc, json, time, shutil, pickle, logging, platform
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, str(Path("dronekit_src").resolve()))
import dronekit
from dronekit import data, infer, score, plots, report
logging.getLogger("sahi").setLevel(logging.WARNING)
try:
    from ultralytics.utils import LOGGER; LOGGER.setLevel(logging.ERROR)   # hides the 'half is deprecated' flood
except Exception:
    pass

ON_KAGGLE = Path("/kaggle").exists()
INPUT = Path(os.environ.get("DK_INPUT", "/kaggle/input"))
DATA = Path(os.environ.get("DK_DATA", "/tmp/VisDrone"))            # images live in /tmp (lost at session end)
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
OUT = WORK / "phase0"
CUDA = torch.cuda.is_available()
DEVICE, HALF = (0, True) if CUDA else ("cpu", False)
FULL_N = int(os.environ.get("DK_FULL_N", "0")) or None             # local CPU test only: "full" section on N images
RUN_FULL = bool(CUDA or FULL_N) and os.environ.get("DK_SKIP_FULL") != "1"   # full val + test-dev run needs the GPU
GPU_NAME = torch.cuda.get_device_name(0) if CUDA else "cpu"
SMOKE_N = 5
YOLO26S_PATTERN = None          # e.g. "yolo26s_visdrone_best.pt" once the teammate's weights are attached
CLASSES = dronekit.VISDRONE_CLASSES

def find(pattern):
    hits = sorted(INPUT.rglob(pattern)) if INPUT.exists() else []
    if len(hits) > 1:
        print(f"  note: {len(hits)} files match {pattern!r}; using {hits[0]}")
    return hits[0] if hits else None

# Zero-shot prompt sets, copied from yoloe-1.ipynb. "descriptive" won on val there (AP50 0.261 vs 0.200 for names).
ZS_PROMPTS = {
    "descriptive": {"pedestrian": "person walking or standing", "people": "person sitting", "bicycle": "bicycle",
                    "car": "car", "van": "van", "truck": "truck", "tricycle": "tricycle",
                    "awning-tricycle": "tricycle with a canopy", "bus": "bus", "motor": "motorcycle"},
    "names": {c: c for c in CLASSES},
}
D2 = "VisDrone2019-DET train → D2: ignore-masked full images + 640/960 px tiles (25% overlap)"
MODELS = {
    "YOLO26m-Tile": dict(weights=find("K1_y26m_tiles/weights/best.pt"), results=find("K1_y26m_tiles/results.csv"),
                         args=find("K1_y26m_tiles/args.yaml"), notebook="visdrone-training", init="yolo26m.pt (COCO)",
                         train_data=D2, train_images="70,216 (6,471 full + 63,745 tiles)"),
    "YOLOE-26m-FT": dict(weights=find("yoloe26m_e2_full.pt"), results=find("e2_full/results.csv"),
                         args=find("e2_full/args.yaml"), notebook="yoloe-2", init="yoloe-26m-seg.pt",
                         train_data=D2, train_images="70,216 (6,471 full + 63,745 tiles)"),
    "YOLOE-26m-LP": dict(weights=find("yoloe26m_e1_linprobe.pt"), results=find("e1_linprobe/results.csv"),
                         args=find("e1_linprobe/args.yaml"), notebook="yoloe-1", init="yoloe-26m-seg.pt",
                         train_data=D2, train_images="70,216 (6,471 full + 63,745 tiles)",
                         note="linear probe: 15,420 of 21,788,308 params trainable (yoloe-1 log)"),
    "YOLOE-26m-ZS": dict(weights="yoloe-26m-seg.pt", prompts="descriptive", notebook="none (Ultralytics release)",
                         init="yoloe-26m-seg.pt", train_data="not trained on VisDrone (zero-shot)", train_images="0"),
    "YOLO26s": dict(weights=find(YOLO26S_PATTERN) if YOLO26S_PATTERN else None, notebook="teammate harness",
                    init="yolo26s.pt (COCO)", train_data="VisDrone2019-DET train, 640 px slices (teammate)",
                    train_images="see teammate notebook"),
}
# Extra val-only row: zero-shot with plain class names, whole image only (resolves which prompts gave 0.261).
EXTRA_VAL_ONLY = {"YOLOE-26m-ZS (names)": dict(weights="yoloe-26m-seg.pt", prompts="names", passes=["full@1536"])}
ENSEMBLE = ("YOLO26m-Tile", "YOLOE-26m-FT")
SETUPS = ["whole1536", "best_sliced"]
PASSES = infer.SETUPS["best_sliced"]["passes"]            # ["s512@1280", "full@1536"]; whole1536 reuses full@1536

print("Python", platform.python_version(), "| torch", torch.__version__, "| CUDA", CUDA, "| GPUs", torch.cuda.device_count())
for k, v in MODELS.items():
    print(f"{k:14s} weights={v['weights']}\n{'':14s} results={v.get('results')}  args={v.get('args')}")
missing = [k for k in ("YOLO26m-Tile", "YOLOE-26m-FT", "YOLOE-26m-LP") if MODELS[k]["weights"] is None]
if missing:
    print("\n!! weights not found for", missing, ": attach the notebook outputs listed at the top (they will be skipped)")

In [ ]:
# --- Data: VisDrone val + test-dev (official annotations) -------------------
SPLITS = {}
for s in ("val", "test-dev"):
    d = DATA / f"VisDrone2019-DET-{s}"
    if not (d / "annotations").exists() and (ON_KAGGLE or s == "val"):
        d = data.download_visdrone(DATA, s)
    if (d / "annotations").exists():
        SPLITS[s] = data.load_visdrone(d, s)
STATS = pd.DataFrame({s: data.split_stats(sp) for s, sp in SPLITS.items()}).T
print(STATS[["images", "sequences", "objects", "objects_per_image", "tiny", "small", "medium_plus", "share_bus"]].round(3))
# expected: val 548 / 76 / 70.7 per image / tiny 0.308 / small 0.377 / bus 0.006;
#                              test-dev 1610 / 61 / 46.6 / 0.364 / 0.313 / 0.039

In [ ]:
# --- Helpers: load a model, run + cache its passes, score every setup -------
def load_model(name, cfg):
    from ultralytics import YOLO, YOLOE
    if cfg.get("prompts"):
        m = YOLOE(str(cfg["weights"]))
        m.set_classes([ZS_PROMPTS[cfg["prompts"]][c] for c in CLASSES])   # one prompt per class, in class order
        cls_map = None
    else:
        m = YOLO(str(cfg["weights"]))                 # 'yoloe' in the filename -> loads as YOLOE
        names = [m.names[i] for i in range(len(m.names))]
        if names == CLASSES:
            cls_map = None
        elif set(names) == set(CLASSES):
            cls_map = {i: CLASSES.index(n) for i, n in enumerate(names)}
        else:
            raise ValueError(f"{name}: model classes {names} are not the VisDrone classes")
    m.predict(np.zeros((640, 640, 3), np.uint8), imgsz=640, half=HALF, device=DEVICE, verbose=False)   # warm-up
    return m, cls_map


def run_models(models, split, cache_dir):
    # One cache per model x split: every raw pass down to conf 0.01, with slice boxes and timings.
    paths = {}
    for name, cfg in models.items():
        if cfg.get("weights") is None:
            continue
        path = cache_dir / f"{name.replace(' ', '_')}__{split.name}"
        if not path.with_suffix(".parquet").exists():
            m, cmap = load_model(name, cfg)
            infer.run_and_cache(m, split, cfg.get("passes", PASSES), path, cuda=CUDA, device=DEVICE, half=HALF,
                                cls_map=cmap, meta=dict(model=name, weights=str(cfg["weights"]),
                                                        prompts=cfg.get("prompts"), device=str(DEVICE),
                                                        gpu=GPU_NAME))
            del m; gc.collect()
            if CUDA: torch.cuda.empty_cache()
        paths[name] = path
    return paths


def save_dets(dets, split, path):
    rows = [np.c_[np.full(len(d), i), d] for i, d in enumerate(dets[r.key] for r in split.images) if len(d)]
    a = np.concatenate(rows) if rows else np.zeros((0, 7))
    df = pd.DataFrame(a[:, 1:], columns=["x1", "y1", "x2", "y2", "score", "cls"]).astype(np.float32)
    df.insert(0, "image", pd.Categorical([split.images[int(i)].key for i in a[:, 0]]))
    df["cls"] = df["cls"].astype(np.int16)
    df.to_parquet(path, compression="zstd", index=False)


def score_everything(caches, splits, out):
    # caches: {split_name: {model: cache path}}. val is processed first so its best-F1 confidence
    # can be reused unchanged on test-dev.
    res_dir, fig_dir, det_dir = out / "results", out / "figures", out / "merged_dets"
    for d in (res_dir, fig_dir, det_dir): d.mkdir(parents=True, exist_ok=True)
    rows, pcs, curves, cms, conf_star, keep = [], [], {}, {}, {}, {}

    def record(name, sname, setup, dets, ms):
        sp = splits[sname]
        m, pc, E = score.evaluate(sp, dets)
        if sname == "val" or (name, setup) not in conf_star:
            conf_star[(name, setup)] = score.best_f1_conf(E) if E is not None else 0.25
        c = conf_star[(name, setup)]
        prc, pr25 = score.pr_at_conf(E, c), score.pr_at_conf(E, 0.25)
        rows.append(dict(model=name, split=sname, setup=setup, n_images=len(sp),
                         precision=prc["precision"], recall=prc["recall"], f1=prc["f1"], conf=c,
                         P_at_0_25=pr25["precision"], R_at_0_25=pr25["recall"],
                         mAP50=m["AP50"], **{"mAP50-95": m["AP"]}, AP75=m.get("AP75"),
                         AP50_tiny=m.get("AP50_tiny"), AP50_small=m.get("AP50_small"),
                         **{"AP50_medium+": m.get("AP50_medium+")}, ms_per_img=round(ms, 1)))
        for k, v in pc.items():
            pcs.append(dict(model=name, split=sname, setup=setup, cls=k, AP50=v))
        if setup == "best_sliced" and E is not None:
            curves[(name, sname)] = score.pr_curves(E)
            cms[(name, sname)] = (score.confusion_matrix(sp, dets, c), c)
        print(f"  {name:22s} {sname:8s} {setup:12s} mAP50={m['AP50']:.4f} mAP50-95={m['AP']:.4f} "
              f"P={prc['precision']:.3f} R={prc['recall']:.3f} @conf {c:.2f}  {ms:.0f} ms")

    for sname in [s for s in ("val", "test-dev") if s in caches]:
        sp = splits[sname]
        for name, path in caches[sname].items():
            raw, meta = infer.load_cache(path)
            for setup in SETUPS:
                if not all(p in raw for p in infer.SETUPS[setup]["passes"]):
                    continue
                dets, mms = infer.apply_setup(raw, sp, setup)
                record(name, sname, setup, dets, infer.setup_ms(meta, setup, mms))
                save_dets(dets, sp, det_dir / f"{name.replace(' ', '_')}__{sname}__{setup}.parquet")
                if setup == "best_sliced" and name in ENSEMBLE:
                    keep[(name, sname)] = (dets, infer.setup_ms(meta, setup, mms))
            del raw; gc.collect()
        if all((n, sname) in keep for n in ENSEMBLE):
            t0 = time.perf_counter()
            dets = infer.ensemble([keep[(n, sname)][0] for n in ENSEMBLE], sp)
            ms = sum(keep[(n, sname)][1] for n in ENSEMBLE) + 1000 * (time.perf_counter() - t0) / len(sp)
            record("Ensemble", sname, "best_sliced", dets, ms)
            save_dets(dets, sp, det_dir / f"Ensemble__{sname}__best_sliced.parquet")

    summary = pd.DataFrame(rows)
    per_class = pd.DataFrame(pcs)
    summary.to_csv(res_dir / "basic_metrics.csv", index=False)
    per_class.to_csv(res_dir / "per_class_ap50_long.csv", index=False)
    for (name, sname), (cm, c) in cms.items():
        pd.DataFrame(cm, index=CLASSES + ["background"], columns=CLASSES + ["background"]).to_csv(
            res_dir / f"confusion_{name.replace(' ', '_')}__{sname}.csv")
    return summary, per_class, curves, cms

In [ ]:
# --- Regression check: dronekit vs the old eval-v2 numbers (CPU, uses the attached evaluation output) ---
# Scores the *old* cached passes from evaluation(1).ipynb with dronekit. Expect an exact match with results_v2.csv:
# same detections in, same protocol -> same AP. Any difference means the package changed the protocol.
OLD_CSV = find("results_v2.csv")
REGRESSION = []
if OLD_CSV is None:
    print("evaluation output not attached: regression check skipped")
else:
    old = pd.read_csv(OLD_CSV)
    for tag in ("K1", "E2", "E1"):
        for sname, sp in SPLITS.items():
            raw = {}
            for p in PASSES:
                f = find(f"{tag}_{sname}_{p}_all.pkl")
                if f is not None:
                    out_, _sec = pickle.load(open(f, "rb"))        # {image id (position): (dets, slice boxes)}
                    raw[p] = {sp.images[i].key: v for i, v in out_.items()}
            for setup, (passes, merge) in {"whole1536": ("full@1536", "NMS-IOU-0.5"),
                                           "best_sliced": ("s512@1280+full@1536", "NMS-IOU-0.6+edge")}.items():
                ref = old[(old["model"] == tag) & (old["split"] == sname) & (old["passes"] == passes) & (old["merge"] == merge)]
                if ref.empty or not all(p in raw for p in infer.SETUPS[setup]["passes"]):
                    continue
                dets, _ = infer.apply_setup(raw, sp, setup)
                m, _, _ = score.evaluate(sp, dets)
                REGRESSION.append(dict(model=tag, split=sname, setup=setup, old_AP50=ref.AP50.iloc[0], new_AP50=round(m["AP50"], 4),
                                       old_AP=ref.AP.iloc[0], new_AP=round(m["AP"], 4)))
            del raw; gc.collect()
    REG = pd.DataFrame(REGRESSION)
    print(REG.to_string(index=False) if len(REG) else "no old caches found (attach eval_v2/cache)")
    if len(REG):
        bad = REG[(REG.old_AP50 - REG.new_AP50).abs() > 0.0011]
        assert bad.empty, f"dronekit does not reproduce eval v2:\n{bad}"
        print("regression check passed: dronekit reproduces eval v2")

In [ ]:
# --- 5-image smoke test (runs on CPU or GPU; finishes the whole pipeline on 5 val images) ---
SMOKE = WORK / "phase0_smoke"
shutil.rmtree(SMOKE, ignore_errors=True)
smoke_split = {"val": SPLITS["val"].subset(SMOKE_N)}
smoke_models = {k: v for k, v in MODELS.items() if v.get("weights") is not None}
t0 = time.time()
smoke_caches = {"val": run_models(smoke_models, smoke_split["val"], SMOKE / "caches")}
s_sum, s_pc, s_curves, s_cms = score_everything(smoke_caches, smoke_split, SMOKE)
for name in list(s_curves)[:1]:
    plots.plot_pr_curves({"val": s_curves[name]}, CLASSES, SMOKE / "figures" / "pr.png", title=f"{name[0]} (smoke)")
    plots.plot_confusion(score.normalize_rows(s_cms[name][0]), CLASSES, SMOKE / "figures" / "cm.png", "smoke", s_cms[name][1])
assert len(s_sum) >= 2 * len(smoke_models), "missing rows"
assert s_sum["mAP50"].between(0, 1).all()
raw0, meta0 = infer.load_cache(next(iter(smoke_caches["val"].values())))
assert set(raw0) == set(PASSES) and len(raw0["full@1536"]) <= SMOKE_N
print(f"\nsmoke test passed in {time.time() - t0:.0f} s ({len(smoke_models)} models)")
s_sum[["model", "setup", "mAP50", "mAP50-95", "precision", "recall", "ms_per_img"]]

## Full run (GPU only): val + test-dev for every model

In [ ]:
# Test-dev rule: each configuration is scored on test-dev once. YOLO26m-Tile, YOLOE-26m-FT and the
# Ensemble repeat configurations already reported; YOLOE-26m-LP and YOLOE-26m-ZS are scored on test-dev for the first time.
CACHES = {}
if RUN_FULL:
    if FULL_N:
        SPLITS = {s: sp.subset(FULL_N) for s, sp in SPLITS.items()}
    for sname, sp in SPLITS.items():
        print(f"== {sname}: {len(sp)} images")
        CACHES[sname] = run_models({k: v for k, v in MODELS.items() if v.get("weights") is not None}, sp, OUT / "caches")
    CACHES["val"].update(run_models(EXTRA_VAL_ONLY, SPLITS["val"], OUT / "caches"))
else:
    print("no GPU: full run skipped (the smoke test above checked the pipeline)")

In [ ]:
# --- Score: whole1536 and best_sliced for every model, plus the Ensemble ------
if RUN_FULL:
    SUMMARY, PER_CLASS, CURVES, CMS = score_everything(CACHES, SPLITS, OUT)
    cols = ["model", "split", "setup", "precision", "recall", "conf", "mAP50", "mAP50-95",
            "AP50_tiny", "AP50_small", "AP50_medium+", "ms_per_img"]
    display(SUMMARY[cols].round(3))

In [ ]:
# --- Per-class AP50: val vs test-dev side by side (best sliced), with GT counts per class ---
if RUN_FULL:
    res = OUT / "results"
    counts = {s: pd.Series(np.concatenate([r.boxes[:, 0] for r in sp.images]).astype(int)).value_counts()
              for s, sp in SPLITS.items()}
    wide = pd.DataFrame(index=CLASSES)
    for s in SPLITS:
        wide[f"n_{s}"] = [int(counts[s].get(k, 0)) for k in range(len(CLASSES))]
        wide[f"share_{s}"] = (wide[f"n_{s}"] / wide[f"n_{s}"].sum()).round(4)
    bs = PER_CLASS[PER_CLASS.setup == "best_sliced"]
    for name in bs.model.unique():
        for s in SPLITS:
            wide[f"{name} {s}"] = bs[(bs.model == name) & (bs.split == s)].set_index("cls").AP50.reindex(CLASSES).values
        if all(f"{name} {s}" in wide for s in ("val", "test-dev")):
            wide[f"{name} drop"] = wide[f"{name} val"] - wide[f"{name} test-dev"]
    wide.index.name = "class"
    wide.to_csv(res / "per_class_val_vs_testdev.csv")
    display(wide.round(3))

In [ ]:
# --- Figures: PR curves, confusion matrices, per-class bars, training curves ---
if RUN_FULL:
    fig = OUT / "figures"
    for name in SUMMARY.model.unique():
        fn = name.replace(" ", "_")
        cv = {s: CURVES[(name, s)] for s in ("val", "test-dev") if (name, s) in CURVES}
        if cv:
            plots.plot_pr_curves(cv, CLASSES, fig / f"pr_{fn}.png", title=f"{name}: PR curves at IoU 0.5 (best sliced)")
        for s in ("val", "test-dev"):
            if (name, s) in CMS:
                cm, c = CMS[(name, s)]
                plots.plot_confusion(score.normalize_rows(cm), CLASSES, fig / f"confusion_{fn}__{s}.png", f"{name}, {s}", c)
        cols_ = [c for c in (f"{name} val", f"{name} test-dev") if c in wide]
        if cols_:
            plots.plot_per_class_bars(wide[cols_].rename(columns=lambda c: c.split(" ")[-1]), fig / f"per_class_{fn}.png",
                                      title=f"{name}: per-class AP50 (best sliced)",
                                      cols=[c.split(" ")[-1] for c in cols_])
    for name, cfg in MODELS.items():
        if cfg.get("results"):
            plots.plot_training_curves(cfg["results"], fig / f"training_{name}.png", title=name)
    print(*sorted(p.name for p in fig.glob("*.png")), sep="\n")

In [ ]:
# --- Training provenance --------------------------------------------------------
prov = []
for name, cfg in MODELS.items():
    w = cfg.get("weights")
    trained_by_us = w and not cfg.get("prompts")        # the zero-shot release was not trained by us
    info = report.read_run(w if Path(str(w)).exists() else None, cfg.get("results"), cfg.get("args")) if trained_by_us else {}
    prov.append(dict(model=name, training_dataset=cfg["train_data"], split="train" if cfg["train_images"] != "0" else "-",
                     image_count=cfg["train_images"], epochs=info.get("epochs_completed"), train_hours=info.get("train_hours"),
                     best_epoch=info.get("best_epoch"), init=cfg["init"], weights_file=Path(str(w)).name if w else "not attached",
                     notebook=cfg["notebook"], note=cfg.get("note", ""), **{k: v for k, v in info.items() if k.startswith("arg_")},
                     ultralytics_version=info.get("ultralytics_version"),
                     **{"val mAP50 (Ultralytics validator)": info.get("best_val_mAP50_ultralytics"),
                        "val mAP50-95 (Ultralytics validator)": info.get("best_val_mAP50-95_ultralytics")}))
PROV = pd.DataFrame(prov)
(OUT / "results").mkdir(parents=True, exist_ok=True)
PROV.to_csv(OUT / "results" / "training_provenance.csv", index=False)
display(PROV)

In [ ]:
# --- Bundle: weights + run files + code next to the results, then the markdown tables ---
if RUN_FULL:
    for name, cfg in MODELS.items():
        d = OUT / "runs" / name; d.mkdir(parents=True, exist_ok=True)
        for k in ("results", "args"):
            if cfg.get(k): shutil.copy(cfg[k], d / Path(cfg[k]).name)
        w = cfg.get("weights")
        if w and Path(str(w)).exists() and not cfg.get("prompts"):
            dst = "yolo26m_tile_visdrone.pt" if name == "YOLO26m-Tile" else Path(w).name   # keep 'yoloe' in YOLOE names
            (OUT / "weights").mkdir(exist_ok=True); shutil.copy(w, OUT / "weights" / dst)
    shutil.copytree("dronekit_src", OUT / "dronekit_src", dirs_exist_ok=True)
    if len(REGRESSION): pd.DataFrame(REGRESSION).to_csv(OUT / "results" / "regression_vs_eval_v2.csv", index=False)
    STATS.to_csv(OUT / "results" / "split_stats.csv")
    cols = ["model", "split", "setup", "precision", "recall", "conf", "mAP50", "mAP50-95",
            "AP50_tiny", "AP50_small", "AP50_medium+", "ms_per_img"]
    md_ = ["# Phase 0 tables (generated)", "", f"dronekit {dronekit.__version__}, GPU: {GPU_NAME}", "",
           "## Basic metrics (precision/recall at the val best-F1 confidence, IoU 0.5)", report.md_table(SUMMARY[cols]), "",
           "## Per-class AP50, best sliced", report.md_table(wide.reset_index()), "",
           "## Training provenance", report.md_table(PROV), "",
           "## Regression vs eval v2", report.md_table(pd.DataFrame(REGRESSION)) if len(REGRESSION) else "not run"]
    (OUT / "results" / "REPORT_tables.md").write_text("\n".join(md_), encoding="utf-8")
    meta = dict(dronekit=dronekit.__version__, torch=torch.__version__, python=platform.python_version(),
                gpu=GPU_NAME, settings=dict(conf=infer.CONF, max_det=infer.MAX_DET, overlap=infer.OVERLAP,
                setups=infer.SETUPS, ensemble_wbf=infer.ENSEMBLE_WBF_THR, max_dets=score.MAX_DETS, areas=score.AREAS))
    import ultralytics, sahi; meta.update(ultralytics=ultralytics.__version__, sahi=sahi.__version__)
    (OUT / "results" / "run_meta.json").write_text(json.dumps(meta, indent=2, default=str))
    !du -sh {OUT} {OUT}/caches {OUT}/merged_dets {OUT}/weights
    print(open(OUT / "results" / "REPORT_tables.md", encoding="utf-8").read())